# 📓 Notas Simulador · Trisectorial Estacional

> Cuaderno de lectura sobre cómo se construyó `src/simulacion/simulador.py`: **el inicio, las dificultades y el fin**.
> Cada 🔖 marca una función que caracteriza el proyecto, con un ejemplo mínimo en Python puro y luego su versión real.

| 🔖 Marcador | Función | Qué resuelve |
|---|---|---|
| 1 | `semanas_de()` | Partir cada season en 13 semanas de datos (S01–S13) |
| 2 | `peso_dia()` · `intensidad()` | Curvas de demanda por sector y país |
| 3 | `entidades_resto()` · `entidades_nuevas()` | Parámetros base de cada entidad |
| 4 | `generar_campana()` | Recorrido del anuncio: inversión → impresiones → clics → conversiones |
| 5 | `ensuciar_campana()` · `fila_de()` | Sembrar errores y registrar su posición exacta |
| 6 | `escribir()` · `bytes_en_memoria()` | Guardar el CSV y alimentar el contador de tamaño |
| 7 | `random.Random(SEMILLA)` | Reproducibilidad |

Todo el cuaderno usa solo la biblioteca estándar de Python (`csv`, `datetime`, `random`), así que corre sin instalar nada.

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd()                      # este cuaderno vive en la raíz de contexto-analisis
sys.path.insert(0, str(RAIZ / "src" / "simulacion"))
import simulador as sim                # importar no genera archivos: main() solo corre desde la terminal

print("Simulador cargado desde:", Path(sim.__file__).relative_to(RAIZ))

---
## 🟢 1 · El inicio

**Punto de partida:** un prototipo de un solo sector (10 restaurantes, 4 semanas de datos de octubre 2026) y una idea más grande:

> *Estudio de comprobación a nivel predictivo por estaciones en base a la economía global, digital híbrida, donde la Alimentación, el Ejercicio y el Comercio se estudian como comportamiento a través de las redes sociales.*

**Decisiones que dieron forma al simulador:**

- 4 sectores: `resto`, `fitness`, `retail`, `tech`, cada uno en su carpeta (`rts-csv`, `ftnss-csv`, `rtl-csv`, `tech-csv`).
- 4 seasons de oct-2025 a sep-2026, cada una con **13 semanas de datos** (S01–S13).
- **Nomenclatura de tiempo:** *sprint de trabajo* (S1–S9) es el calendario de trabajo del proyecto; *semana de datos* (S01–S13) es el tiempo que miden los datos dentro de cada season.
- **Mismos índices** en todos los sectores; cambia qué es una conversión y cuándo es temporada alta.
- Los archivos crudos traen **solo datos capturados**; los KPIs los calcula el pipeline.
- **Errores sembrados a propósito**, con una lista de respuestas para comprobar la limpieza.

### 🔖 1 · `semanas_de()` — el calendario

Un trimestre no tiene exactamente 13 semanas (91 días): tiene 90, 91 o 92. La regla es simple: las semanas de datos S01–S12 duran 7 días y **la S13 absorbe lo que sobre**.

**Ejemplo básico:**

In [ ]:
import datetime as dt

inicio, fin = dt.date(2026, 1, 1), dt.date(2026, 3, 31)      # season2: 90 días
dias_totales = (fin - inicio).days + 1
dias_s13 = dias_totales - 12 * 7
print(f"season2 tiene {dias_totales} días → 12 semanas de datos de 7 + S13 de {dias_s13} días")

**En el proyecto:** el generador devuelve `(semana, fecha_inicio, fecha_fin)` para cada season. En los crudos esa semana de datos se guarda en la columna `sprint`.

In [ ]:
for season in sim.SEASONS:
    semanas = list(sim.semanas_de(season))
    s13 = semanas[-1]
    print(f"season{season}: {semanas[0][1]} → {s13[2]} · S13 dura {(s13[2] - s13[1]).days + 1} días")

### 🔖 2 · `peso_dia()` e `intensidad()` — la temporada alta de cada sector

Cada día recibe un **peso de demanda** = peso del mes × evento especial (si lo hay) × rango especial (si lo hay).
La **intensidad** de una semana de datos es el promedio de sus días: `1.0` es una semana normal, `1.5` una semana 50 % más fuerte.

**Ejemplo básico:**

In [ ]:
peso_mes = {11: 1.6, 12: 1.8}            # retail: noviembre y diciembre fuertes
eventos = {dt.date(2025, 11, 28): 3.0}   # Black Friday

def peso(d):
    return peso_mes.get(d.month, 1.0) * eventos.get(d, 1.0)

semana_bf = [dt.date(2025, 11, 24) + dt.timedelta(n) for n in range(7)]
print("Intensidad semana de Black Friday:", round(sum(peso(d) for d in semana_bf) / 7, 2))

**En el proyecto:** la misma semana, por sector y por país. Así se ven las **similitudes** (picos de fin de año) y las **diferencias** (fitness cae en diciembre; Bogotá tiene otra curva).

In [ ]:
f_ini, f_fin = dt.date(2025, 11, 26), dt.date(2025, 12, 2)    # semana de Black Friday + Cyber Monday
for sector in ["resto", "retail", "fitness", "tech"]:
    usa = sim.intensidad(sector, "USA", f_ini, f_fin)
    col = sim.intensidad(sector, "Colombia", f_ini, f_fin)
    print(f"{sector:8s} USA {usa:4.2f} · Colombia {col:4.2f}")

### 🔖 3 · Las entidades y sus parámetros base

Cada entidad tiene su "personalidad": inversión semanal, CPM, CTR, tasa de conversión, ticket y costo variable, más una **tendencia** (mejora o empeora durante el año).

- **Restaurantes:** se heredan del archivo original (`RESTO_BASE`), para dar continuidad al prototipo.
- **Fitness, retail y tech:** 10 entidades ficticias por sector, con parámetros sorteados dentro de rangos realistas (`PARAMS`).

In [ ]:
for sector, rangos in sim.PARAMS.items():
    print(f"{sector:8s} ticket {rangos['ticket']} · costo variable {rangos['cv']}")
print()
print("Ejemplo restaurante R010:", sim.RESTO_BASE["R010"])

### 🔖 4 · `generar_campana()` — el recorrido del anuncio

Es el corazón del simulador. Cada fila sigue el embudo **inversión → impresiones → alcance → clics → conversiones**, y la intensidad de la semana de datos mueve cada etapa:

| Etapa | Regla | Por qué |
|---|---|---|
| Inversión | sube con la intensidad | En temporada alta se invierte más |
| CPM | sube con la intensidad | Más competencia encarece la puja |
| Clics | `impresiones × CTR` | El CTR mejora en temporada alta |
| Conversiones | `clics × tasa de conversión` | Cambia según el sector |

**Ejemplo básico** (una semana normal y una semana pico):

In [ ]:
def semana(inversion_base, cpm_base, ctr, cr, intensidad):
    inversion = inversion_base * (1 + 0.5 * (intensidad - 1))
    cpm = cpm_base * (1 + 0.3 * (intensidad - 1))
    impresiones = int(inversion / cpm * 1000)
    clics = int(impresiones * ctr * (1 + 0.4 * (intensidad - 1)))
    conversiones = int(clics * cr * intensidad ** 0.8)
    return dict(inversion=round(inversion), impresiones=impresiones, clics=clics, conversiones=conversiones)

print("normal:", semana(5000, 20, 0.02, 0.06, 1.0))
print("pico  :", semana(5000, 20, 0.02, 0.06, 1.8))

**Lo que cambia por sector** (mismos índices, distinto significado):

- 🍔 **resto** · conversión = pedido, reserva o cupón.
- 🛒 **retail** · compra en línea; en los picos **baja el ticket** por los descuentos.
- 🏋️ **fitness** · membresía; `ticket_promedio` = cuota mensual y `meses_retencion_promedio` permite calcular el LTV. Quien entra en enero dura menos.
- 💻 **tech** · `leads → leads_calificados → conversiones` (clientes cerrados). Puede haber semanas de datos con 0 clientes.

In [ ]:
import random

sim.rng = random.Random(1)                                 # semilla local solo para el ejemplo
ents = sim.entidades_nuevas("tech", sim.TECH[:1])
fila = sim.generar_campana("tech", ents)[1][0]
print({k: fila[k] for k in ["id_entidad", "sprint", "inversion_pauta", "clics", "leads", "leads_calificados", "conversiones"]})

---
## 🟠 2 · Las dificultades

Cada tropiezo dejó una regla que ahora forma parte del proyecto.

### ⚠️ Dificultad 1 · Los saltos de línea invisibles (CRLF)

Al reasignar las fechas de `coffee_sales_raw.csv`, el archivo se leyó con `read_text()`, que **convierte `\r\n` en `\n` sin avisar**. El script no encontró ninguna línea que partir y reescribió el archivo con otros saltos de línea. Se restauró desde el respaldo.

**Regla:** para leer un CSV tal cual viene, abrirlo con `newline=""`.

In [ ]:
import io

contenido = "a,b\r\n1,2\r\n"
normalizado = io.StringIO(contenido, newline=None).read()   # lo que hace read_text()
tal_cual = io.StringIO(contenido, newline="").read()        # lo que hace open(..., newline="")
print("read_text():", repr(normalizado))
print("newline='' :", repr(tal_cual))

### ⚠️ Dificultad 2 · Un simulador que se lee a sí mismo

La primera versión tomaba los parámetros de los restaurantes del mismo archivo que luego **sobrescribía**. La segunda corrida habría leído datos ya simulados (y ensuciados), y el resultado cambiaría en cada ejecución.

**Regla:** los parámetros de origen se calculan una vez y se congelan en el código (`RESTO_BASE`). Una prueba sencilla lo confirma: dos corridas seguidas dan archivos idénticos.

In [ ]:
import hashlib

def huella(texto):
    return hashlib.md5(texto.encode()).hexdigest()[:10]

r1, r2 = random.Random(42), random.Random(42)
print(huella(str([r1.random() for _ in range(5)])), "=", huella(str([r2.random() for _ in range(5)])))

### ⚠️ Dificultad 3 · Las filas se corrían al insertar errores

Al insertar un duplicado o una fila vacía, todas las filas de abajo bajan un lugar, y los números ya anotados en la lista de anomalías quedaban mal.

**Regla:** primero se hacen todas las inserciones y **al final** se anota la posición, buscando cada fila por identidad (`is`) con `fila_de()`.

In [ ]:
filas = [{"id": "A"}, {"id": "B"}, {"id": "C"}]
marcada = filas[2]                     # la fila "C" recibe un error
filas.insert(0, {"id": ""})            # después se inserta una fila vacía arriba

print("Posición anotada antes de insertar: 4")
print("Posición real con fila_de():", sim.fila_de(filas, marcada))

### ⚠️ Dificultad 4 · El cronómetro medía otra cosa

El primer archivo del registro marcaba **1.2 s** y los demás 0.02 s. No era el archivo: era `import pandas`, que ocurría dentro del cronómetro la primera vez.

**Regla:** importar al inicio y cronometrar solo el trabajo que se quiere medir.

In [ ]:
import time

t0 = time.perf_counter()
import json                            # ya cargado: casi 0 s
print(f"segunda importación: {time.perf_counter() - t0:.6f} s")

### ⚠️ Dificultad 5 · Retail perdía dinero todo el año

Con la primera calibración, retail tenía ROAS de 1.1 a 1.8: con un margen de 30–45 % eso significa pérdidas incluso en Black Friday. **No era realista.** Se bajó el CPM y se subió la tasa de conversión hasta lograr ROAS de 2.8 a 4.4.

**Regla:** el punto de equilibrio es `ROAS = 1 / (1 − costo variable)`.

In [ ]:
for cv in [0.40, 0.55, 0.70]:
    print(f"costo variable {cv:.0%} → ROAS mínimo para no perder: {1 / (1 - cv):.2f}")

### ⚠️ Dificultad 6 · Fitness parecía no rentable (y no era un error)

Fitness salía con ROI negativo, pero aquí la simulación estaba bien. Lo que pasaba era que se medía **un solo mes de cuota**. Una membresía paga muchos meses: hay que usar el **LTV**.

In [ ]:
cuota, meses, cv, cpa = 80, 9, 0.30, 90
roi_un_mes = (cuota * (1 - cv) - cpa) / cpa
roi_ltv = (cuota * meses * (1 - cv) - cpa) / cpa
print(f"ROI con 1 mes: {roi_un_mes:+.2f} · ROI con LTV ({meses} meses): {roi_ltv:+.2f}")

### ⚠️ Dificultad 7 · ¿Qué es una conversión en tech?

Si `conversiones` fueran los leads, la fórmula `ingresos = conversiones × ticket` daría ingresos por formularios llenados. Se definió **`conversiones` = clientes cerrados**, y `leads` / `leads_calificados` quedaron como etapas previas del embudo. Así la fórmula vale igual en los 4 sectores.

---
### 🔖 5 · `ensuciar_campana()` — errores a propósito, con lista de respuestas

Cada season recibe una mezcla distinta de 7 u 8 errores, más un duplicado exacto y una fila vacía. Todo queda en `src/simulacion/anomalias_sembradas.csv`, que después servirá para comprobar si el pipeline los detecta.

In [ ]:
import csv
from collections import Counter

with open(RAIZ / "src" / "simulacion" / "anomalias_sembradas.csv", encoding="utf-8", newline="") as f:
    anomalias = list(csv.DictReader(f))

print(len(anomalias), "anomalías sembradas\n")
for tipo, n in Counter(a["tipo"] for a in anomalias).most_common():
    print(f"{n:3d}  {tipo}")

### 🔖 6 · `escribir()` — el contador de tamaño

Cada archivo que se guarda deja una fila en `src/registro_tamano.csv`: filas, bytes en disco, bytes en memoria (si hay pandas) y segundos. Sirve para vigilar que el sistema no se vuelva lento a medida que crecen los datos.

In [ ]:
with open(RAIZ / "src" / "registro_tamano.csv", encoding="utf-8", newline="") as f:
    registro = list(csv.DictReader(f))

total = sum(int(r["bytes_disco"]) for r in registro)
print(f"{len(registro)} archivos · {total / 1024:.1f} KB en disco en total\n")
for r in registro[:5]:
    print(f"{r['archivo']:42s} {r['filas']:>4s} filas · {int(r['bytes_disco']) / 1024:5.1f} KB · {r['segundos']} s")

### 🔖 7 · `random.Random(SEMILLA)` — reproducibilidad

Toda la aleatoriedad sale de un generador con semilla fija (`SEMILLA = 20251001`). Con la misma semilla, el simulador produce **exactamente los mismos archivos**, y eso permite comparar resultados entre sesiones.

In [ ]:
print("Semilla del proyecto:", sim.SEMILLA)
a, b = random.Random(sim.SEMILLA), random.Random(sim.SEMILLA)
print([round(a.uniform(0, 1), 3) for _ in range(3)])
print([round(b.uniform(0, 1), 3) for _ in range(3)])

---
## 🔵 3 · El fin (por ahora)

**Lo que quedó construido:**

```
src/
├── rts-csv/    restaurantes_raw.csv + season1–4_campana_resto_raw.csv (+ .md)
├── ftnss-csv/  fitness_raw.csv      + season1–4_campana_fitness_raw.csv
├── rtl-csv/    retail_raw.csv       + season1–4_campana_retail_raw.csv
├── tech-csv/   tech_raw.csv         + season1–4_campana_tech_raw.csv
├── simulacion/ simulador.py + anomalias_sembradas.csv
└── registro_tamano.csv
```

**Cómo se comportan los mercados** (sin contar las filas sembradas):

| Sector | Pico | ROI en el pico | ROI en el valle | Rasgo |
|---|---|---|---|---|
| 🍔 Restaurantes | season1 | 2.38 | 1.59 | Los más rentables |
| 🛒 Retail | season1 | 0.69 | 0.07 | Volumen alto, margen bajo |
| 🏋️ Fitness | season2 | −0.20 (1 mes) | −0.66 | ROI anual con LTV: 3.71 |
| 💻 Tech | season1 y 4 | 0.31 | 0.10 | Embudo largo |

**Para regenerar todo:**

```bash
py src/simulacion/simulador.py
```

**Siguiente paso:** el pipeline M1 en `src/modelado/` (Definición → Recopilación → Exploración → Preprocesamiento), que limpiará estos archivos y usará `anomalias_sembradas.csv` para comprobar que detectó cada error.